In [1]:
# GPU check — confirm XGBoost can see the GPU before running.
import subprocess, xgboost as xgb

_smi = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True
)
print(f"GPU:  {_smi.stdout.strip() or 'none detected — request a GPU node'}")
print(f"XGB:  {xgb.build_info()}")  # confirms this XGBoost build has CUDA support


GPU:  NVIDIA A100-PCIE-40GB, 40960 MiB
XGB:  {'BUILTIN_PREFETCH_PRESENT': True, 'CUDA_VERSION': [12, 9], 'DEBUG': False, 'GCC_VERSION': [10, 3, 1], 'GLIBC_VERSION': [2, 28], 'MM_PREFETCH_PRESENT': True, 'NCCL_VERSION': [2, 29, 2], 'THRUST_VERSION': [2, 8, 2], 'USE_CUDA': True, 'USE_DLOPEN_NCCL': True, 'USE_FEDERATED': True, 'USE_NCCL': True, 'USE_NVCOMP': False, 'USE_OPENMP': True, 'USE_RMM': False, 'libxgboost': '/home/mb6477/.conda/envs/hackathon_2026/lib/python3.11/site-packages/xgboost/lib/libxgboost.so'}


In [2]:
import os
import geopandas as gpd
import numpy as np
import pandas as pd
from datetime import datetime
from xgboost import XGBClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import classification_report


In [3]:
# ! pip install xgboost

In [4]:
# ── paths ────────────────────────────────────────────────────────
# GDB_DIR     : statewide NJ parcels geodatabase (Esri GDB format)
# OBM_NJ      : statewide OpenBuildings footprints geopackage (~2.7M footprints)
# LABELS_1808 : manually cleaned lookuptable for tract 1808 — source of positive training pairs
# LABELS_1219 : manually cleaned parcels for tract 1219 — second batch of training labels
# UNCLEANED_PARCELS : toy-problem spatial join file; provides parcel geometries for the labeled set
# OUT_BASE    : root directory under which timestamped run folders are created
GDB_DIR           = "/scratch/gpfs/GVILLARI/hackathon_2026/FULL_NJ/Statewide_Parcels_MODIV.gdb"
OBM_NJ            = "/scratch/gpfs/GVILLARI/hackathon_2026/FULL_NJ/obm_nj.gpkg"
LABELS_1808       = "/scratch/gpfs/GVILLARI/mb6477/parcelAI/cleaned_lookuptable.gpkg"
LABELS_1219       = "/scratch/gpfs/GVILLARI/mb6477/parcelAI/1219_CLEAN_PARCELS.gpkg"
UNCLEANED_PARCELS = "/scratch/gpfs/GVILLARI/hackathon_2026/TOY_PROBLEM/spatial_join_for_cleaning.gpkg"
OUT_BASE          = "/scratch/gpfs/GVILLARI/mb6477/parcelAI/outputs_XGboost"

In [5]:
# ── parameters ──────────────────────────────────────────────────
# TEST_MUNIS        : which municipality codes to run prediction on.
#                     Parsed from the leading digits of PAMS_PIN (e.g. "1205_0001_00001" → "1205").
#                     Add more codes here to expand the prediction area without changing any
#                     other logic. Remove both to predict on ALL statewide parcels (slow).
TEST_MUNIS = {"1205", "1216"}

# N_FOLDS           : number of folds for GroupKFold spatial cross-validation.
#                     Groups are PAMS_PINs, so each fold holds out whole parcels — this prevents
#                     data leakage where the same parcel's pairs appear in both train and test.
#                     5 is standard; reduce to 3 if labeled set is small (<200 parcels per fold).
N_FOLDS = 5

# ── XGBoost hyperparameters ───────────────────────────────────────
# N_ESTIMATORS      : number of boosting rounds (trees).
#                     200 is a reasonable starting point; with boosting you often need fewer
#                     trees than a Random Forest. Raise toward 500–1000 for final runs, but
#                     watch for overfitting — pair larger values with a smaller LEARNING_RATE.
# LEARNING_RATE     : step size shrinkage (eta). Lower = more conservative, needs more trees.
#                     0.1 is a solid default; try 0.05 with more estimators for a final model.
# MAX_DEPTH         : maximum depth of each tree. Controls model complexity / overfitting.
#                     6 is the XGBoost default; 3–4 for more regularisation, 8+ for complex signal.
# SUBSAMPLE         : row subsampling per boosting round (stochastic gradient boosting).
# COLSAMPLE_BYTREE  : column (feature) subsampling per tree. Both add regularisation.
# random_state      : fixed seed so results are reproducible across runs with the same data.
N_ESTIMATORS     = 200
LEARNING_RATE    = 0.1
MAX_DEPTH        = 6
SUBSAMPLE        = 0.9
COLSAMPLE_BYTREE = 0.9
RANDOM_STATE     = 42

# CLASS IMBALANCE   : XGBoost has no class_weight="balanced". The equivalent is
#                     scale_pos_weight = (#negative pairs) / (#positive pairs), which scales the
#                     gradient of the rare positive (correct-match) class. It is computed from the
#                     training labels just before the model is built (see the training cell), so it
#                     adapts automatically as the labeled set changes.

# NOTE: these are just for manual checking and don't really matter for outcomes
# CONFIDENCE_THRESH : model probability below this → needs_review = True in all output layers.
#                     0.8 means the model must be at least 80% confident to call a pair a match
#                     without flagging it for human review. Lower to catch more potential errors;
#                     raise to reduce review queue at the cost of missing some ambiguous cases.
CONFIDENCE_THRESH = 0.8

# STRONG_CAND_THRESH: overlap_frac threshold used to count "strong candidates" per parcel.
#                     A footprint is a strong candidate for a parcel if it overlaps more than
#                     this fraction of its own area with that parcel. Used as feature
#                     n_strong_candidates — tells the model how contested a parcel is.
STRONG_CAND_THRESH = 0.3

In [6]:
# ── timestamped output directory ───────────────────────────────────
# Each run writes to its own subdirectory so nothing is ever overwritten.
run_ts  = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
OUT_PATH = os.path.join(OUT_BASE, run_ts)
os.makedirs(OUT_PATH, exist_ok=True)
print(f"Output directory: {OUT_PATH}")

Output directory: /scratch/gpfs/GVILLARI/mb6477/parcelAI/outputs_XGboost/2026-06-02_14-10-02


In [7]:
# ── print all parameters ────────────────────────────────────────
print("\n── Run parameters ───────────────────────────────────────")
print(f"  TEST_MUNIS         = {TEST_MUNIS}")
print(f"  N_FOLDS            = {N_FOLDS}   (GroupKFold spatial CV; groups = PAMS_PIN)")
print(f"  N_ESTIMATORS       = {N_ESTIMATORS}  (XGBoost boosting rounds)")
print(f"  LEARNING_RATE      = {LEARNING_RATE}  (eta; step size shrinkage)")
print(f"  MAX_DEPTH          = {MAX_DEPTH}    (max tree depth)")
print(f"  SUBSAMPLE          = {SUBSAMPLE}  (row subsampling per round)")
print(f"  COLSAMPLE_BYTREE   = {COLSAMPLE_BYTREE}  (feature subsampling per tree)")
print(f"  scale_pos_weight   = auto  (neg/pos ratio; compensates for negative-pair imbalance)")
print(f"  RANDOM_STATE       = {RANDOM_STATE}   (RNG seed for reproducibility)")
print(f"  CONFIDENCE_THRESH  = {CONFIDENCE_THRESH}  (below this → needs_review = True)")
print(f"  STRONG_CAND_THRESH = {STRONG_CAND_THRESH}  (overlap_frac floor for n_strong_candidates)")
print("────────────────────────────────────────────────────────────────\n")


── Run parameters ───────────────────────────────────────
  TEST_MUNIS         = {'1216', '1205'}
  N_FOLDS            = 5   (GroupKFold spatial CV; groups = PAMS_PIN)
  N_ESTIMATORS       = 200  (XGBoost boosting rounds)
  LEARNING_RATE      = 0.1  (eta; step size shrinkage)
  MAX_DEPTH          = 6    (max tree depth)
  SUBSAMPLE          = 0.9  (row subsampling per round)
  COLSAMPLE_BYTREE   = 0.9  (feature subsampling per tree)
  scale_pos_weight   = auto  (neg/pos ratio; compensates for negative-pair imbalance)
  RANDOM_STATE       = 42   (RNG seed for reproducibility)
  CONFIDENCE_THRESH  = 0.8  (below this → needs_review = True)
  STRONG_CAND_THRESH = 0.3  (overlap_frac floor for n_strong_candidates)
────────────────────────────────────────────────────────────────



In [8]:
# ── load labels ───────────────────────────────────────────────
# Both label files must contain PAMS_PIN, footprint_ids (comma-separated fp id string),
# and method (used as a quality filter — rows with null method are excluded).
print("Loading ground truth labels...")
labels_1808 = gpd.read_file(LABELS_1808)[["PAMS_PIN", "footprint_ids", "method"]]
labels_1219 = gpd.read_file(LABELS_1219)[["PAMS_PIN", "footprint_ids", "method"]]

labels_1808 = labels_1808.dropna(subset=["method"])
labels_1219 = labels_1219.dropna(subset=["method"])

labels = pd.concat([labels_1808, labels_1219], ignore_index=True)
labels = labels.drop_duplicates(subset=["PAMS_PIN"], keep="first")
labeled_pins = set(labels["PAMS_PIN"])

print(f"Labeled parcels (training): {len(labeled_pins)} "
      f"(1808: {len(labels_1808)}, 1219: {len(labels_1219)})")

# Build the positive set: (PAMS_PIN, fp_id) pairs that are known correct matches.
labels["fp_id_list"] = labels["footprint_ids"].apply(
    lambda x: x.split(",") if isinstance(x, str) else []
)
pairs = labels[["PAMS_PIN", "fp_id_list"]].explode("fp_id_list")
pairs = pairs[pairs["fp_id_list"].notna() & (pairs["fp_id_list"] != "")]
pairs.columns = ["PAMS_PIN", "fp_id"]
pairs["fp_id"] = pairs["fp_id"].astype(str).str.strip()
positive_set = set(zip(pairs["PAMS_PIN"], pairs["fp_id"]))
print(f"Positive training pairs:    {len(pairs)}")

Loading ground truth labels...
Labeled parcels (training): 1348 (1808: 748, 1219: 600)
Positive training pairs:    1283


In [9]:
# ── load statewide parcels, filter to test municipalities ───────────────────
# Parcels are read from the statewide GDB and filtered by the leading muni code in PAMS_PIN.
# To expand to more municipalities, add codes to TEST_MUNIS above.
# To run statewide, remove the TEST_MUNIS filter entirely (expect very long overlay runtime).
print(f"\nLoading statewide parcels filtered to {TEST_MUNIS}...")
parcels_nj = gpd.read_file(
    GDB_DIR,
    layer="Cad_parcel_mod4",
    columns=["PAMS_PIN", "geometry"]
).to_crs("EPSG:3424")

parcels_nj["muni"] = parcels_nj["PAMS_PIN"].str.split("_").str[0]
parcels_test   = parcels_nj[parcels_nj["muni"].isin(TEST_MUNIS)].drop(columns="muni").copy()
parcels_unseen = parcels_test[~parcels_test["PAMS_PIN"].isin(labeled_pins)].copy()
unseen_pins    = set(parcels_unseen["PAMS_PIN"])

print(f"Parcels in {TEST_MUNIS}:     {len(parcels_test)}")
print(f"  Labeled (in training set): {len(parcels_test[parcels_test['PAMS_PIN'].isin(labeled_pins)])}")
print(f"  Unseen (will predict):     {len(parcels_unseen)}")


Loading statewide parcels filtered to {'1216', '1205'}...


/home/mb6477/.conda/envs/hackathon_2026/lib/python3.11/site-packages/pyogrio/raw.py:200: RuntimeWarning: organizePolygons() received a polygon with more than 100 parts. The processing may be really slow.  You can skip the processing by setting METHOD=SKIP, or only make it analyze counter-clock wise parts by setting METHOD=ONLY_CCW if you can assume that the outline of holes is counter-clock wise defined
  return ogr_read(


Parcels in {'1216', '1205'}:     42360
  Labeled (in training set): 0
  Unseen (will predict):     42360


In [10]:
# ── load labeled parcel geometries ───────────────────────────────
# The toy-problem file is used for labeled geometries rather than the GDB because these are
# the exact geometries the labels were drawn against — consistency matters for the overlay.
parcels_labeled = gpd.read_file(
    UNCLEANED_PARCELS,
    columns=["PAMS_PIN", "geometry"]
).to_crs("EPSG:3424")
parcels_labeled = parcels_labeled[parcels_labeled["PAMS_PIN"].isin(labeled_pins)]
print(f"  Labeled parcel geometries loaded: {len(parcels_labeled)}")

# Combine labeled + unseen test parcels for a single overlay pass.
# Group features (entropy, total candidate coverage) need the full combined set so that
# footprints straddling labeled and unseen parcels are described correctly.
parcels_all = pd.concat(
    [parcels_labeled[["PAMS_PIN", "geometry"]],
     parcels_unseen[["PAMS_PIN", "geometry"]]],
    ignore_index=True
)
parcels_all = gpd.GeoDataFrame(parcels_all, geometry="geometry", crs="EPSG:3424")
print(f"\nCombined parcel set for overlay: {len(parcels_all)} "
      f"({len(parcels_labeled)} labeled + {len(parcels_unseen)} unseen)")

  Labeled parcel geometries loaded: 1348

Combined parcel set for overlay: 43708 (1348 labeled + 42360 unseen)


In [11]:
# ── parcel geometry features ───────────────────────────────────
parcels_all["parcel_area"]       = parcels_all.geometry.area
parcels_all["parcel_perim"]      = parcels_all.geometry.length
parcels_all["parcel_centroid_x"] = parcels_all.geometry.centroid.x
parcels_all["parcel_centroid_y"] = parcels_all.geometry.centroid.y

In [12]:
# ── load OBM footprints clipped to test area ─────────────────────────
# Clip to the bounding box of parcels_all rather than loading all 2.7M statewide footprints.
# This is the key performance optimisation for the test run.
# For a full statewide run: replace bbox=bbox with no spatial filter and adjust OUT_BASE.
print("\nLoading OBM footprints clipped to test area...")
bbox = tuple(parcels_all.total_bounds)  # (minx, miny, maxx, maxy)
obm  = gpd.read_file(OBM_NJ, bbox=bbox).to_crs("EPSG:3424")
print(f"OBM footprints in test area: {len(obm)} (vs 2,703,732 statewide)")

obm["fp_area"]       = obm.geometry.area
obm["fp_perim"]      = obm.geometry.length
obm["fp_compact"]    = (4 * np.pi * obm["fp_area"]) / (obm["fp_perim"] ** 2)
obm["fp_centroid_x"] = obm.geometry.centroid.x
obm["fp_centroid_y"] = obm.geometry.centroid.y
obm["id"]            = obm["id"].astype(str).str.strip()


Loading OBM footprints clipped to test area...
OBM footprints in test area: 135937 (vs 2,703,732 statewide)


### GENERATE PARAMETERS FROM GEOSPATIAL INFO

In [13]:
# ── overlay ─────────────────────────────────────────────────
# Intersection of footprints × parcels produces one row per (fp, parcel) candidate pair.
# All features are derived from this table.
print("\nBuilding overlay...")
ov = gpd.overlay(
    obm[["id", "geometry", "fp_area", "fp_perim", "fp_compact",
         "fp_centroid_x", "fp_centroid_y"]],
    parcels_all[["PAMS_PIN", "geometry", "parcel_area", "parcel_perim",
                 "parcel_centroid_x", "parcel_centroid_y"]],
    how="intersection"
)

ov["id"]            = ov["id"].astype(str).str.strip()
ov["piece_area"]    = ov.geometry.area
ov["overlap_frac"]  = ov["piece_area"] / ov["fp_area"]       # fraction of fp that falls in parcel
ov["area_ratio"]    = ov["fp_area"] / ov["parcel_area"]       # relative size
ov["area_diff"]     = abs(ov["fp_area"] - ov["parcel_area"])  # absolute size mismatch
ov["centroid_dist"] = np.sqrt(
    (ov["parcel_centroid_x"] - ov["fp_centroid_x"]) ** 2 +
    (ov["parcel_centroid_y"] - ov["fp_centroid_y"]) ** 2
)
ov["n_fps_for_parcel"] = ov.groupby("PAMS_PIN")["id"].transform("count")
ov["n_parcels_for_fp"] = ov.groupby("id")["PAMS_PIN"].transform("count")
ov["overlap_rank"]     = ov.groupby("PAMS_PIN")["overlap_frac"].rank(ascending=False)
ov["area_rank"]        = ov.groupby("PAMS_PIN")["fp_area"].rank(ascending=False)
ov["frac_of_best"]     = ov["overlap_frac"] / ov.groupby("PAMS_PIN")["overlap_frac"].transform("max")

print(f"Total candidate pairs: {len(ov)}")


Building overlay...
Total candidate pairs: 55938


In [14]:
# ── GROUP 1: footprint partition quality ────────────────────────────
# These features describe how "cleanly" a footprint sits inside one parcel vs. being split
# across many. A footprint with high entropy is spread across many parcels — likely a
# segmentation error or a large building straddling a boundary.
# GPU fp_split_entropy vectorized — replaces groupby("id").apply(python_fn) which ran
# one Python call per footprint and was the slowest pandas op in the notebook.
ov["_fr"]    = ov["overlap_frac"].clip(lower=1e-9)
_denom       = ov.groupby("id")["_fr"].transform("sum")
_p           = ov["_fr"] / _denom
ov["_plogp"] = -(_p * np.log(_p))
ov["fp_split_entropy"] = ov.groupby("id")["_plogp"].transform("sum")
ov.drop(columns=["_fr", "_plogp"], inplace=True)

ov["fp_max_overlap"]        = ov.groupby("id")["overlap_frac"].transform("max")
ov["fp_is_best_parcel"]     = (ov["overlap_frac"] == ov["fp_max_overlap"]).astype(int)
ov["fp_gap_from_best"]      = ov["fp_max_overlap"] - ov["overlap_frac"]
ov["fp_abs_area_elsewhere"] = ov["fp_area"] * (1.0 - ov["overlap_frac"])


In [15]:
# ── GROUP 2: parcel coverage ───────────────────────────────────
# These features describe how well the footprint candidates collectively cover the parcel.
# A parcel with low total coverage may be vacant or have no footprint at all.
ov["fp_covers_parcel_frac"] = ov["piece_area"] / ov["parcel_area"]
ov["parcel_total_candidate_coverage"] = (
    ov.groupby("PAMS_PIN")["piece_area"].transform("sum") / ov["parcel_area"]
)
# GPU vectorized n_strong_candidates — replaces the lambda transform
# which ran a Python function per group.
ov["_above"]              = (ov["overlap_frac"] > STRONG_CAND_THRESH).astype("int8")
ov["n_strong_candidates"] = ov.groupby("PAMS_PIN")["_above"].transform("sum")
ov.drop(columns=["_above"], inplace=True)


#### TRAIN XGBOOST

In [16]:
# ── label and train/predict split ───────────────────────────────
# GPU vectorized label assignment — replaces apply(axis=1) which ran a Python
# lambda per row (the single slowest pandas op in the notebook).
_keys        = list(zip(ov["PAMS_PIN"].to_numpy(), ov["id"].to_numpy()))
ov["correct"]    = np.fromiter(
    (1 if k in positive_set else 0 for k in _keys),
    count=len(_keys), dtype="int8"
)
ov["is_labeled"] = ov["PAMS_PIN"].isin(labeled_pins)

ov_labeled = ov[ov["is_labeled"]].copy()
ov_unseen  = ov[~ov["is_labeled"]].copy()

print(f"\nTraining set:   {len(ov_labeled)} pairs "
      f"(pos: {ov_labeled['correct'].sum()}, "
      f"neg: {(ov_labeled['correct']==0).sum()})")
print(f"Prediction set: {len(ov_unseen)} pairs "
      f"across {ov_unseen['PAMS_PIN'].nunique()} parcels")



Training set:   1668 pairs (pos: 1283, neg: 385)
Prediction set: 54270 pairs across 39694 parcels


In [17]:
# ── features ─────────────────────────────────────────────
FEATURES = [
    "fp_area", "fp_perim", "fp_compact",
    "parcel_area", "parcel_perim",
    "centroid_dist", "area_ratio", "area_diff",
    "overlap_frac", "piece_area", "frac_of_best",
    "n_fps_for_parcel", "n_parcels_for_fp",
    "overlap_rank", "area_rank",
    "fp_split_entropy", "fp_max_overlap", "fp_is_best_parcel",
    "fp_gap_from_best", "fp_abs_area_elsewhere",
    "fp_covers_parcel_frac", "parcel_total_candidate_coverage",
    "n_strong_candidates",
]

X      = ov_labeled[FEATURES].fillna(0)
y      = ov_labeled["correct"]
groups = ov_labeled["PAMS_PIN"]

In [18]:
# ── spatial cross-validation ───────────────────────────────────
# GroupKFold ensures whole parcels are held out together — no pair from a parcel
# can appear in both train and test within a fold. This gives an honest estimate
# of how the model will generalise to the unseen municipalities.
gkf = GroupKFold(n_splits=N_FOLDS)

# scale_pos_weight is XGBoost's equivalent of class_weight="balanced": the ratio of
# negative to positive training pairs, so the rare correct-match class is up-weighted.
# GPU computed per fold on the train split only — avoids leakage into the held-out fold.
def _make_model(spw):
    return XGBClassifier(
        n_estimators     = N_ESTIMATORS,
        learning_rate    = LEARNING_RATE,
        max_depth        = MAX_DEPTH,
        subsample        = SUBSAMPLE,
        colsample_bytree = COLSAMPLE_BYTREE,
        scale_pos_weight = spw,
        objective        = "binary:logistic",
        eval_metric      = "logloss",
        device           = "cuda",            # GPU — moves all tree building to the GPU
        random_state     = RANDOM_STATE,
        verbosity        = 0,
    )

# GPU single CV loop — one fit per fold, predict_proba on the held-out fold, derive
# labels from probability. Replaces the double cross_val_predict + per-fold refit
# loop (~16 fits total) with 5 fits, one per fold.
print(f"\nRunning {N_FOLDS}-fold spatial CV on labeled data...")
cv_proba = np.zeros(len(y), dtype=float)

for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups)):
    _pos = int(y.iloc[train_idx].sum())
    _neg = int((y.iloc[train_idx] == 0).sum())
    m    = _make_model(spw=_neg / max(_pos, 1))
    m.fit(X.iloc[train_idx], y.iloc[train_idx])
    cv_proba[test_idx] = m.predict_proba(X.iloc[test_idx])[:, 1]
    fold_acc     = ((cv_proba[test_idx] >= 0.5).astype(int) == y.iloc[test_idx].to_numpy()).mean()
    fold_parcels = ov_labeled.iloc[test_idx]["PAMS_PIN"].nunique()
    print(f"  Fold {fold+1}: {fold_parcels} parcels, accuracy {fold_acc:.3f}")

cv_pred = (cv_proba >= 0.5).astype(int)
ov_labeled["cv_pred"]  = cv_pred
ov_labeled["cv_proba"] = cv_proba

print("\n── Overall CV results ──")
print(classification_report(y, cv_pred))

# ── train final model on all labeled data ────────────────────────────
# CV above was for diagnostics only. The final model sees all labeled pairs so it
# has maximum signal before predicting on the unseen municipalities.
print("\nTraining final model on all labeled data...")
n_pos = int((y == 1).sum())
n_neg = int((y == 0).sum())
print(f"scale_pos_weight = {n_neg/n_pos:.2f}  (neg {n_neg} / pos {n_pos})")
model = _make_model(spw=n_neg / max(n_pos, 1))
model.fit(X, y)
print("Feature importances:")
for f, i in sorted(zip(FEATURES, model.feature_importances_), key=lambda x: -x[1]):
    print(f"  {f}: {i:.3f}")

# ── predict on unseen parcels ──────────────────────────────────
print(f"\nPredicting on {ov_unseen['PAMS_PIN'].nunique()} unseen parcels...")
ov_unseen = ov_unseen.copy()
ov_unseen["predicted"]    = model.predict(ov_unseen[FEATURES].fillna(0))
ov_unseen["confidence"]   = model.predict_proba(ov_unseen[FEATURES].fillna(0))[:, 1]
ov_unseen["needs_review"] = ov_unseen["confidence"] < CONFIDENCE_THRESH



Running 5-fold spatial CV on labeled data...
  Fold 1: 230 parcels, accuracy 0.982
  Fold 2: 257 parcels, accuracy 0.973
  Fold 3: 257 parcels, accuracy 0.970
  Fold 4: 257 parcels, accuracy 0.967
  Fold 5: 257 parcels, accuracy 0.970

── Overall CV results ──
              precision    recall  f1-score   support

           0       0.93      0.95      0.94       385
           1       0.98      0.98      0.98      1283

    accuracy                           0.97      1668
   macro avg       0.96      0.96      0.96      1668
weighted avg       0.97      0.97      0.97      1668


Training final model on all labeled data...
scale_pos_weight = 0.30  (neg 385 / pos 1283)
Feature importances:
  piece_area: 0.613
  fp_covers_parcel_frac: 0.062
  area_rank: 0.051
  fp_perim: 0.042
  fp_area: 0.038
  area_diff: 0.020
  frac_of_best: 0.019
  n_strong_candidates: 0.018
  overlap_frac: 0.016
  fp_max_overlap: 0.013
  fp_split_entropy: 0.013
  centroid_dist: 0.012
  n_fps_for_parcel: 0.010
  a

#### SAVE THE OUTPUT

In [19]:
# ── 1. parcels_by_outcome ────────────────────────────────────
# CV outcome for labeled parcels; 'unseen' for everything else.
parcel_summary = ov_labeled.groupby("PAMS_PIN").apply(lambda g: pd.Series({
    "n_candidates":     len(g),
    "n_correct":        g["correct"].sum(),
    "n_predicted_pos":  (g["cv_pred"] == 1).sum(),
    "n_true_pos":       ((g["cv_pred"]==1) & (g["correct"]==1)).sum(),
    "n_false_pos":      ((g["cv_pred"]==1) & (g["correct"]==0)).sum(),
    "n_false_neg":      ((g["cv_pred"]==0) & (g["correct"]==1)).sum(),
    "avg_confidence":   g.loc[g["cv_pred"]==1, "cv_proba"].mean(),
    "predicted_fp_ids": ",".join(g.loc[g["cv_pred"]==1, "id"].tolist()),
})).reset_index()

/tmp/ipykernel_1914886/1158108166.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  parcel_summary = ov_labeled.groupby("PAMS_PIN").apply(lambda g: pd.Series({


In [20]:
def categorize(row):
    if row["n_false_pos"] > 0 and row["n_false_neg"] > 0:
        return "both_errors"
    elif row["n_false_pos"] > 0:
        return "false_positive"
    elif row["n_false_neg"] > 0:
        return "false_negative"
    elif pd.isna(row["avg_confidence"]) or row["avg_confidence"] < CONFIDENCE_THRESH:
        return "low_confidence"
    else:
        return "correct"

parcel_summary["outcome"] = parcel_summary.apply(categorize, axis=1)

parcel_out = parcels_all[["PAMS_PIN", "geometry"]].merge(
    parcel_summary, on="PAMS_PIN", how="left"
)
parcel_out["outcome"] = parcel_out["outcome"].fillna("unseen")

gpd.GeoDataFrame(parcel_out, geometry="geometry", crs="EPSG:3424").to_file(
    f"{OUT_PATH}/parcels_by_outcome.gpkg", driver="GPKG"
)
print("saved parcels_by_outcome.gpkg")
print(parcel_out["outcome"].value_counts())

saved parcels_by_outcome.gpkg
outcome
unseen            42450
correct            1161
low_confidence       57
false_negative       22
false_positive       16
both_errors           2
Name: count, dtype: int64


In [21]:
# ── 2. footprints_by_match_status ──────────────────────────────
# CV diagnostics on labeled parcels only — footprint polygons.
fp_labeled = ov_labeled.merge(
    obm[["id", "geometry"]].rename(columns={"geometry": "fp_geometry"}),
    on="id", how="left"
)
fp_labeled["match_status"] = "true_negative"
fp_labeled.loc[(fp_labeled["correct"]==1) & (fp_labeled["cv_pred"]==1), "match_status"] = "true_positive"
fp_labeled.loc[(fp_labeled["correct"]==0) & (fp_labeled["cv_pred"]==1), "match_status"] = "false_positive"
fp_labeled.loc[(fp_labeled["correct"]==1) & (fp_labeled["cv_pred"]==0), "match_status"] = "false_negative"

gpd.GeoDataFrame(
    fp_labeled[["PAMS_PIN", "id", "match_status", "correct",
                "cv_pred", "cv_proba", "overlap_frac", "piece_area",
                "centroid_dist", "fp_geometry"]],
    geometry="fp_geometry", crs="EPSG:3424"
).to_file(f"{OUT_PATH}/footprints_by_match_status.gpkg", driver="GPKG")
print("\nsaved footprints_by_match_status.gpkg")
print(fp_labeled["match_status"].value_counts())


saved footprints_by_match_status.gpkg
match_status
true_positive     1257
true_negative      365
false_negative      26
false_positive      20
Name: count, dtype: int64


In [22]:
# ── 3. hard_cases_intersections ───────────────────────────────
# Intersection slivers for CV errors and low-confidence pairs — useful for manual review
# and for diagnosing systematic failure modes in QGIS.
hard = ov_labeled[
    (ov_labeled["correct"] != ov_labeled["cv_pred"]) |
    (ov_labeled["cv_proba"] < CONFIDENCE_THRESH)
].copy()
hard["match_status"] = "true_negative"
hard.loc[(hard["correct"]==1) & (hard["cv_pred"]==1), "match_status"] = "true_positive"
hard.loc[(hard["correct"]==0) & (hard["cv_pred"]==1), "match_status"] = "false_positive"
hard.loc[(hard["correct"]==1) & (hard["cv_pred"]==0), "match_status"] = "false_negative"

gpd.GeoDataFrame(
    hard[["PAMS_PIN", "id", "match_status", "correct",
          "cv_pred", "cv_proba", "overlap_frac", "piece_area",
          "centroid_dist", "geometry"]],
    geometry="geometry", crs="EPSG:3424"
).to_file(f"{OUT_PATH}/hard_cases_intersections.gpkg", driver="GPKG")
print(f"\nsaved hard_cases_intersections.gpkg ({len(hard)} pairs)")


saved hard_cases_intersections.gpkg (425 pairs)


In [23]:
# ── 4. predicted_assignments ─────────────────────────────────
# All parcels with assigned footprint IDs.
# source='manual' → ground truth from the cleaned label files (confidence pinned to 1.0)
# source='model'  → XGBoost prediction on the unseen parcels in TEST_MUNIS
manual_assignments = (
    pairs.groupby("PAMS_PIN")
    .agg(predicted_fp_ids=("fp_id", lambda x: ",".join(x)),
         n_predicted=("fp_id", "count"))
    .reset_index()
)
manual_assignments["avg_confidence"] = 1.0
manual_assignments["source"]         = "manual"
manual_assignments["needs_review"]   = False

model_assigned = (
    ov_unseen[ov_unseen["predicted"] == 1]
    .sort_values("confidence", ascending=False)
    .groupby("PAMS_PIN")
    .agg(
        predicted_fp_ids=("id", lambda x: ",".join(x)),
        n_predicted=("id", "count"),
        avg_confidence=("confidence", "mean")
    )
    .reset_index()
)
model_assigned["source"]       = "model"
model_assigned["needs_review"] = model_assigned["avg_confidence"] < CONFIDENCE_THRESH

model_unassigned = pd.DataFrame({
    "PAMS_PIN": [p for p in unseen_pins
                 if p not in set(model_assigned["PAMS_PIN"])]
})
model_unassigned["predicted_fp_ids"] = None
model_unassigned["n_predicted"]      = 0
model_unassigned["avg_confidence"]   = None
model_unassigned["source"]           = "model"
model_unassigned["needs_review"]     = False

results_full = pd.concat(
    [manual_assignments, model_assigned, model_unassigned],
    ignore_index=True
)
results_full = parcels_all[["PAMS_PIN", "geometry"]].merge(
    results_full, on="PAMS_PIN", how="left"
)

assert len(results_full) == len(parcels_all), \
    f"PAMS_PIN mismatch: {len(results_full)} vs {len(parcels_all)}"

gpd.GeoDataFrame(results_full, geometry="geometry", crs="EPSG:3424").to_file(
    f"{OUT_PATH}/predicted_assignments.gpkg", driver="GPKG"
)
print(f"\nsaved predicted_assignments.gpkg ({len(results_full)} parcels)")
print(results_full.groupby("source").agg(
    parcels      =("PAMS_PIN",      "count"),
    assigned     =("n_predicted",   lambda x: (x > 0).sum()),
    needs_review =("needs_review",  "sum")
))

/tmp/ipykernel_1914886/2132615703.py:39: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results_full = pd.concat(



saved predicted_assignments.gpkg (43708 parcels)
        parcels  assigned needs_review
source                                
manual     1202      1202            0
model     42360     38116          641


In [24]:
# ── 5. predicted_footprints_unseen ──────────────────────────────
# Footprint polygons for all model-predicted assignments on unseen parcels.
# Load this layer in QGIS to visually inspect what the model assigned.
# Style by 'confidence' (gradient) or 'needs_review' (binary flag).
fp_unseen = (
    ov_unseen[ov_unseen["predicted"] == 1]
    .merge(
        obm[["id", "geometry"]].rename(columns={"geometry": "fp_geometry"}),
        on="id", how="left"
    )
)

gpd.GeoDataFrame(
    fp_unseen[[
        "PAMS_PIN", "id",
        "confidence", "needs_review",
        "overlap_frac", "fp_covers_parcel_frac",
        "fp_split_entropy", "n_parcels_for_fp",
        "piece_area", "centroid_dist",
        "fp_geometry"
    ]],
    geometry="fp_geometry",
    crs="EPSG:3424"
).to_file(f"{OUT_PATH}/predicted_footprints_unseen.gpkg", driver="GPKG")

print(f"\nsaved predicted_footprints_unseen.gpkg")
print(f"  Unique footprints assigned: {fp_unseen['id'].nunique()}")
print(f"  Parcels covered:            {fp_unseen['PAMS_PIN'].nunique()}")
print(f"  Needs review (conf < {CONFIDENCE_THRESH}):  {fp_unseen['needs_review'].sum()}")
print(f"\nAll outputs written to: {OUT_PATH}")


saved predicted_footprints_unseen.gpkg
  Unique footprints assigned: 30704
  Parcels covered:            38116
  Needs review (conf < 0.8):  1173

All outputs written to: /scratch/gpfs/GVILLARI/mb6477/parcelAI/outputs_XGboost/2026-06-02_14-10-02


#### PREDICT FULL NEW JERSEY

In [25]:
import time, warnings
from tqdm.notebook import tqdm

warnings.filterwarnings("ignore")

# ── NJ output folder — separate subfolder inside OUT_BASE ─────────────────────
# Uses its own timestamp so it never mixes with the test-muni run.
nj_ts   = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
OUT_NJ  = os.path.join(OUT_BASE, f"full_nj_{nj_ts}")
CKPT_NJ = os.path.join(OUT_NJ, "county_checkpoints")
os.makedirs(OUT_NJ,  exist_ok=True)
os.makedirs(CKPT_NJ, exist_ok=True)
print(f"NJ output folder: {OUT_NJ}")

# save the trained model now so the county loop can reload it if the kernel restarts mid-run
MODEL_NJ = os.path.join(OUT_NJ, "xgb_final_model.json")
model.save_model(MODEL_NJ)
print(f"Model saved:      {MODEL_NJ}")

BBOX_BUFFER = 150  # metres buffer around each county bbox when clipping OBM footprints
                   # keeps buildings that straddle county boundaries from being missed

def compute_features_xgb(ov):
    """Compute all 23 XGBoost features on an overlay DataFrame. Fully vectorized."""
    ov = ov.copy()
    ov["id"]            = ov["id"].astype(str).str.strip()
    ov["piece_area"]    = ov.geometry.area
    ov["overlap_frac"]  = ov["piece_area"] / ov["fp_area"]
    ov["area_ratio"]    = ov["fp_area"]  / ov["parcel_area"]
    ov["area_diff"]     = (ov["fp_area"] - ov["parcel_area"]).abs()
    ov["centroid_dist"] = np.sqrt(
        (ov["parcel_centroid_x"] - ov["fp_centroid_x"])**2 +
        (ov["parcel_centroid_y"] - ov["fp_centroid_y"])**2)
    ov["n_fps_for_parcel"] = ov.groupby("PAMS_PIN")["id"].transform("count")
    ov["n_parcels_for_fp"] = ov.groupby("id")["PAMS_PIN"].transform("count")
    ov["overlap_rank"]     = ov.groupby("PAMS_PIN")["overlap_frac"].rank(ascending=False)
    ov["area_rank"]        = ov.groupby("PAMS_PIN")["fp_area"].rank(ascending=False)
    ov["frac_of_best"]     = ov["overlap_frac"] / ov.groupby("PAMS_PIN")["overlap_frac"].transform("max")
    # fp_split_entropy — vectorized Shannon entropy (same logic as training cells above)
    ov["_fr"] = ov["overlap_frac"].clip(lower=1e-9)
    _p        = ov["_fr"] / ov.groupby("id")["_fr"].transform("sum")
    ov["_plogp"]           = -(_p * np.log(_p))
    ov["fp_split_entropy"] = ov.groupby("id")["_plogp"].transform("sum")
    ov.drop(columns=["_fr", "_plogp"], inplace=True)
    ov["fp_max_overlap"]        = ov.groupby("id")["overlap_frac"].transform("max")
    ov["fp_is_best_parcel"]     = (ov["overlap_frac"] == ov["fp_max_overlap"]).astype(int)
    ov["fp_gap_from_best"]      = ov["fp_max_overlap"] - ov["overlap_frac"]
    ov["fp_abs_area_elsewhere"] = ov["fp_area"] * (1.0 - ov["overlap_frac"])
    ov["fp_covers_parcel_frac"] = ov["piece_area"] / ov["parcel_area"]
    ov["parcel_total_candidate_coverage"] = (
        ov.groupby("PAMS_PIN")["piece_area"].transform("sum") / ov["parcel_area"])
    ov["_above"]              = (ov["overlap_frac"] > STRONG_CAND_THRESH).astype("int8")
    ov["n_strong_candidates"] = ov.groupby("PAMS_PIN")["_above"].transform("sum")
    ov.drop(columns=["_above"], inplace=True)
    return ov.reset_index(drop=True)

# ── load all statewide NJ parcels ─────────────────────────────────────────────
# Strip labeled_pins — those already have manual assignments from the label files.
print("\nLoading statewide NJ parcels...")
t0 = time.time()
nj_parcels = gpd.read_file(GDB_DIR, layer="Cad_parcel_mod4",
                            columns=["PAMS_PIN", "geometry"]).to_crs("EPSG:3424")
nj_parcels = (nj_parcels
              .dropna(subset=["PAMS_PIN", "geometry"])
              .drop_duplicates(subset=["PAMS_PIN"])
              .reset_index(drop=True))
nj_parcels = nj_parcels[~nj_parcels["PAMS_PIN"].isin(labeled_pins)].reset_index(drop=True)
nj_parcels["county"] = nj_parcels["PAMS_PIN"].str.split("_").str[0].str[:2]
counties = sorted(nj_parcels["county"].unique())
print(f"  {len(nj_parcels):,} unseen parcels  |  {len(counties)} counties  |  {time.time()-t0:.1f}s")
print(nj_parcels["county"].value_counts().sort_index().to_string())


NJ output folder: /scratch/gpfs/GVILLARI/mb6477/parcelAI/outputs_XGboost/full_nj_2026-06-02_14-53-05
Model saved:      /scratch/gpfs/GVILLARI/mb6477/parcelAI/outputs_XGboost/full_nj_2026-06-02_14-53-05/xgb_final_model.json

Loading statewide NJ parcels...
  3,473,532 unseen parcels  |  21 counties  |  40.8s
county
01    173564
02    299587
03    218425
04    200644
05    143206
06     70572
07    186186
08    125783
09    146437
10     54733
11    141974
12    275000
13    264122
14    180546
15    421548
16    132326
17     32624
18    132038
19     76400
20    149402
21     48415


In [26]:
# ── county-by-county prediction with checkpointing ────────────────────────────
# If the cell crashes or is interrupted: just re-run — completed counties are skipped.

# reload model if the kernel was restarted after Cell A
if "model" not in dir() or not hasattr(model, "predict"):
    from xgboost import XGBClassifier
    model = XGBClassifier()
    model.load_model(MODEL_NJ)
    print(f"Reloaded model from {MODEL_NJ}")

all_results = []
all_fp_rows = []
loop_t0     = time.time()

for ci, county in enumerate(tqdm(counties, desc="NJ counties")):
    ckpt_file = os.path.join(CKPT_NJ, f"county_{county}.csv")
    county_t0 = time.time()

    # skip if checkpoint already exists
    if os.path.exists(ckpt_file):
        cached = pd.read_csv(ckpt_file, dtype={"PAMS_PIN": str, "county": str})
        all_results.append(cached)
        print(f"  [{ci+1}/{len(counties)}] county {county}: checkpoint loaded "
              f"({(cached['n_predicted'] > 0).sum():,} assigned)")
        continue

    # ── parcels ───────────────────────────────────────────────────────────────
    county_parcels = (nj_parcels[nj_parcels["county"] == county]
                      .drop(columns="county")
                      .drop_duplicates(subset=["PAMS_PIN"])
                      .copy().reset_index(drop=True))
    county_parcels["parcel_area"]       = county_parcels.geometry.area
    county_parcels["parcel_perim"]      = county_parcels.geometry.length
    county_parcels["parcel_centroid_x"] = county_parcels.geometry.centroid.x
    county_parcels["parcel_centroid_y"] = county_parcels.geometry.centroid.y

    # ── OBM footprints clipped to county bbox + buffer ────────────────────────
    minx, miny, maxx, maxy = county_parcels.total_bounds
    bbox_buf   = (minx - BBOX_BUFFER, miny - BBOX_BUFFER,
                  maxx + BBOX_BUFFER, maxy + BBOX_BUFFER)
    t1         = time.time()
    county_obm = gpd.read_file(OBM_NJ, bbox=bbox_buf).to_crs("EPSG:3424")
    county_obm = county_obm.drop_duplicates(subset=["id"]).reset_index(drop=True)
    if len(county_obm) == 0:
        print(f"  [{ci+1}/{len(counties)}] county {county}: no OBM footprints — skipping")
        continue
    county_obm["fp_area"]       = county_obm.geometry.area
    county_obm["fp_perim"]      = county_obm.geometry.length
    county_obm["fp_compact"]    = (4 * np.pi * county_obm["fp_area"]) / (county_obm["fp_perim"]**2)
    county_obm["fp_centroid_x"] = county_obm.geometry.centroid.x
    county_obm["fp_centroid_y"] = county_obm.geometry.centroid.y
    county_obm["id"]            = county_obm["id"].astype(str).str.strip()
    print(f"  [{ci+1}/{len(counties)}] county {county}: "
          f"{len(county_parcels):,} parcels  {len(county_obm):,} footprints  "
          f"OBM {time.time()-t1:.1f}s")

    # ── overlay ───────────────────────────────────────────────────────────────
    t1 = time.time()
    try:
        ov = gpd.overlay(
            county_obm[["id", "geometry", "fp_area", "fp_perim", "fp_compact",
                         "fp_centroid_x", "fp_centroid_y"]],
            county_parcels[["PAMS_PIN", "geometry", "parcel_area", "parcel_perim",
                             "parcel_centroid_x", "parcel_centroid_y"]],
            how="intersection")
    except Exception as e:
        print(f"    overlay failed ({e}) — skipping county {county}")
        continue
    if len(ov) == 0:
        print(f"    empty overlay — skipping")
        continue
    print(f"    overlay: {len(ov):,} pairs  {time.time()-t1:.1f}s")

    # ── features ─────────────────────────────────────────────────────────────
    t1 = time.time()
    ov = compute_features_xgb(ov)
    print(f"    features: {time.time()-t1:.1f}s")

    # ── XGBoost prediction (GPU) ──────────────────────────────────────────────
    t1             = time.time()
    X_pred         = ov[FEATURES].fillna(0)
    ov["confidence"]   = model.predict_proba(X_pred)[:, 1]
    ov["predicted"]    = (ov["confidence"] >= 0.5).astype(int)
    ov["needs_review"] = ov["confidence"] < CONFIDENCE_THRESH
    print(f"    XGB predict (GPU): {time.time()-t1:.3f}s")

    # ── aggregate to parcel level ─────────────────────────────────────────────
    assigned = (ov[ov["predicted"] == 1]
        .sort_values("confidence", ascending=False)
        .groupby("PAMS_PIN")
        .agg(predicted_fp_ids=("id",         lambda x: ",".join(x)),
             n_predicted      =("id",         "count"),
             avg_confidence   =("confidence", "mean"))
        .reset_index())
    assigned["needs_review"] = assigned["avg_confidence"] < CONFIDENCE_THRESH

    unassigned_pins = set(county_parcels["PAMS_PIN"]) - set(assigned["PAMS_PIN"])
    unassigned      = pd.DataFrame({"PAMS_PIN": list(unassigned_pins)})
    unassigned["predicted_fp_ids"] = None
    unassigned["n_predicted"]      = 0
    unassigned["avg_confidence"]   = None
    unassigned["needs_review"]     = False

    county_result           = pd.concat([assigned, unassigned], ignore_index=True)
    county_result["county"] = county
    county_result.to_csv(ckpt_file, index=False)
    all_results.append(county_result)

    # keep footprint rows for the final spatial layer
    all_fp_rows.append(ov[ov["predicted"] == 1][[
        "PAMS_PIN", "id", "confidence", "needs_review",
        "overlap_frac", "piece_area", "centroid_dist",
        "fp_split_entropy", "frac_of_best"]].copy())

    n_assigned = (county_result["n_predicted"] > 0).sum()
    elapsed    = time.time() - county_t0
    remaining  = elapsed / (ci + 1) * (len(counties) - ci - 1)
    print(f"    done: {n_assigned:,}/{len(county_parcels):,} assigned  "
          f"{int(assigned['needs_review'].sum()):,} need review  "
          f"{elapsed:.0f}s  ~{remaining/60:.0f} min left")

print(f"\nAll {len(counties)} counties done in {(time.time()-loop_t0)/60:.1f} min")


NJ counties:   0%|          | 0/21 [00:00<?, ?it/s]

  [1/21] county 01: 173,564 parcels  145,465 footprints  OBM 1.9s
    overlay: 159,615 pairs  5.6s
    features: 1.1s
    XGB predict (GPU): 0.310s
    done: 105,609/173,564 assigned  3,799 need review  11s  ~4 min left
  [2/21] county 02: 299,587 parcels  422,745 footprints  OBM 5.5s
    overlay: 338,626 pairs  12.9s
    features: 2.3s
    XGB predict (GPU): 0.347s
    done: 197,033/299,587 assigned  5,538 need review  26s  ~4 min left
  [3/21] county 03: 218,425 parcels  318,283 footprints  OBM 4.3s
    overlay: 215,956 pairs  8.7s
    features: 1.6s
    XGB predict (GPU): 0.258s
    done: 157,776/218,425 assigned  2,655 need review  18s  ~2 min left
  [4/21] county 04: 200,644 parcels  288,065 footprints  OBM 3.9s
    overlay: 219,697 pairs  7.5s
    features: 1.7s
    XGB predict (GPU): 0.263s
    done: 168,961/200,644 assigned  2,836 need review  16s  ~1 min left
  [5/21] county 05: 143,206 parcels  88,987 footprints  OBM 2.0s
    overlay: 152,139 pairs  5.5s
    features: 0.9s
  

In [27]:
# ── combine county results and save final NJ outputs ─────────────────────────
print("Combining county results...")
results             = pd.concat(all_results, ignore_index=True)
results["PAMS_PIN"] = results["PAMS_PIN"].astype(str)

n_assigned   = int((results["n_predicted"] > 0).sum())
n_unassigned = int((results["n_predicted"] == 0).sum())
n_review     = int(results["needs_review"].sum())
print(f"  total parcels: {len(results):,}")
print(f"  assigned:      {n_assigned:,}  ({100*n_assigned/len(results):.1f}%)")
print(f"  unassigned:    {n_unassigned:,}")
print(f"  needs review:  {n_review:,}  (confidence < {CONFIDENCE_THRESH})")

# 1. flat CSV — fast to load and query without GIS software
results.to_csv(os.path.join(OUT_NJ, "nj_xgb_prediction_summary.csv"), index=False)
print("\nsaved nj_xgb_prediction_summary.csv")

# 2. parcel polygons with assignment attributes — open in QGIS, style by n_predicted
t1 = time.time()
results_geo = nj_parcels[["PAMS_PIN", "geometry"]].merge(results, on="PAMS_PIN", how="left")
gpd.GeoDataFrame(results_geo, geometry="geometry", crs="EPSG:3424").to_file(
    os.path.join(OUT_NJ, "nj_xgb_predicted_assignments.gpkg"), driver="GPKG")
print(f"saved nj_xgb_predicted_assignments.gpkg  ({time.time()-t1:.1f}s)")

# 3. footprint polygons for every assigned pair — style by confidence or needs_review
if all_fp_rows:
    t1      = time.time()
    fp_all  = pd.concat(all_fp_rows, ignore_index=True)
    print(f"\nLoading OBM geometries for footprint layer ({len(fp_all):,} assigned pairs)...")
    obm_geom       = gpd.read_file(OBM_NJ, columns=["id", "geometry"])
    obm_geom["id"] = obm_geom["id"].astype(str).str.strip()
    fp_geo         = fp_all.merge(obm_geom, on="id", how="left")
    gpd.GeoDataFrame(fp_geo, geometry="geometry", crs="EPSG:3424").to_file(
        os.path.join(OUT_NJ, "nj_xgb_predicted_footprints.gpkg"), driver="GPKG")
    print(f"saved nj_xgb_predicted_footprints.gpkg  ({time.time()-t1:.1f}s)")

print(f"\nAll NJ outputs in: {OUT_NJ}")
print(f"  nj_xgb_prediction_summary.csv         — flat table, one row per parcel")
print(f"  nj_xgb_predicted_assignments.gpkg     — parcel polygons with assignments")
print(f"  nj_xgb_predicted_footprints.gpkg      — footprint polygons, styled by confidence")
print(f"  county_checkpoints/                   — {len(counties)} county CSVs (resume support)")


Combining county results...
  total parcels: 3,473,532
  assigned:      2,667,534  (76.8%)
  unassigned:    805,998
  needs review:  53,383  (confidence < 0.8)

saved nj_xgb_prediction_summary.csv
saved nj_xgb_predicted_assignments.gpkg  (38.5s)

Loading OBM geometries for footprint layer (2,881,455 assigned pairs)...
saved nj_xgb_predicted_footprints.gpkg  (44.8s)

All NJ outputs in: /scratch/gpfs/GVILLARI/mb6477/parcelAI/outputs_XGboost/full_nj_2026-06-02_14-53-05
  nj_xgb_prediction_summary.csv         — flat table, one row per parcel
  nj_xgb_predicted_assignments.gpkg     — parcel polygons with assignments
  nj_xgb_predicted_footprints.gpkg      — footprint polygons, styled by confidence
  county_checkpoints/                   — 21 county CSVs (resume support)
